In [2]:
import numpy as np
import random

# ============================================================
#  PEAS - Data Structures and Fringes that define the Agent environment
# ============================================================

ROW_COUNT = 6
COLUMN_COUNT = 7
PLAYER_PIECE = 1
AI_PIECE = 2
EMPTY = 0
WINDOW_LENGTH = 4

def create_board():
    """Create a 6x7 board filled with zeros"""
    board = np.zeros((ROW_COUNT, COLUMN_COUNT))
    return board

def drop_piece(board, row, col, piece):
    """Drop a piece (player or AI) into the board"""
    board[row][col] = piece

def is_valid_location(board, col):
    """Check if the column is a valid move (not full)"""
    return board[ROW_COUNT-1][col] == 0

def get_next_open_row(board, col):
    """Get the next open row for a move"""
    for r in range(ROW_COUNT):
        if board[r][col] == 0:
            return r

def print_board(board):
    """Display the board state"""
    print(np.flip(board, 0))

def get_valid_locations(board):
    """Get all columns where a move is possible"""
    valid_locations = [col for col in range(COLUMN_COUNT) if is_valid_location(board, col)]
    return valid_locations

def winning_move(board, piece):
    """Check if there is a winning move"""
    # Horizontal check
    for c in range(COLUMN_COUNT - 3):
        for r in range(ROW_COUNT):
            if all([board[r][c+i] == piece for i in range(WINDOW_LENGTH)]):
                return True
    # Vertical check
    for c in range(COLUMN_COUNT):
        for r in range(ROW_COUNT - 3):
            if all([board[r+i][c] == piece for i in range(WINDOW_LENGTH)]):
                return True
    # Positively sloped diagonals
    for c in range(COLUMN_COUNT - 3):
        for r in range(ROW_COUNT - 3):
            if all([board[r+i][c+i] == piece for i in range(WINDOW_LENGTH)]):
                return True
    # Negatively sloped diagonals
    for c in range(COLUMN_COUNT - 3):
        for r in range(3, ROW_COUNT):
            if all([board[r-i][c+i] == piece for i in range(WINDOW_LENGTH)]):
                return True
    return False

def is_terminal_node(board):
    """Check if the game has ended (win or draw)"""
    return winning_move(board, PLAYER_PIECE) or winning_move(board, AI_PIECE) or len(get_valid_locations(board)) == 0


# ============================================================
#  Implementation of the Min-Max Algorithm (Without Alpha-Beta Pruning)
# ============================================================

def minimax(board, depth, maximizingPlayer):
    """Minimax algorithm without alpha-beta pruning"""
    valid_locations = get_valid_locations(board)
    is_terminal = is_terminal_node(board)

    if depth == 0 or is_terminal:
        if is_terminal:
            if winning_move(board, AI_PIECE):
                return (None, 100000000)
            elif winning_move(board, PLAYER_PIECE):
                return (None, -100000000)
            else:
                return (None, 0)
        else:
            return (None, score_position(board, AI_PIECE))

    if maximizingPlayer:
        value = -np.inf
        column = random.choice(valid_locations)
        for col in valid_locations:
            row = get_next_open_row(board, col)
            temp_board = board.copy()
            drop_piece(temp_board, row, col, AI_PIECE)
            new_score = minimax(temp_board, depth-1, False)[1]
            if new_score > value:
                value = new_score
                column = col
        return column, value
    else:
        value = np.inf
        column = random.choice(valid_locations)
        for col in valid_locations:
            row = get_next_open_row(board, col)
            temp_board = board.copy()
            drop_piece(temp_board, row, col, PLAYER_PIECE)
            new_score = minimax(temp_board, depth-1, True)[1]
            if new_score < value:
                value = new_score
                column = col
        return column, value


# ============================================================
#  Implementation of the Alpha-Beta Pruning (With Alpha-Beta Pruning)
# ============================================================

def minimax_alpha_beta(board, depth, alpha, beta, maximizingPlayer):
    """Minimax algorithm with alpha-beta pruning"""
    valid_locations = get_valid_locations(board)
    is_terminal = is_terminal_node(board)

    if depth == 0 or is_terminal:
        if is_terminal:
            if winning_move(board, AI_PIECE):
                return (None, 100000000)
            elif winning_move(board, PLAYER_PIECE):
                return (None, -100000000)
            else:
                return (None, 0)
        else:
            return (None, score_position(board, AI_PIECE))

    if maximizingPlayer:
        value = -np.inf
        column = random.choice(valid_locations)
        for col in valid_locations:
            row = get_next_open_row(board, col)
            temp_board = board.copy()
            drop_piece(temp_board, row, col, AI_PIECE)
            new_score = minimax_alpha_beta(temp_board, depth-1, alpha, beta, False)[1]
            if new_score > value:
                value = new_score
                column = col
            alpha = max(alpha, value)
            if alpha >= beta:
                break  # Alpha cut-off
        return column, value
    else:
        value = np.inf
        column = random.choice(valid_locations)
        for col in valid_locations:
            row = get_next_open_row(board, col)
            temp_board = board.copy()
            drop_piece(temp_board, row, col, PLAYER_PIECE)
            new_score = minimax_alpha_beta(temp_board, depth-1, alpha, beta, True)[1]
            if new_score < value:
                value = new_score
                column = col
            beta = min(beta, value)
            if alpha >= beta:
                break  # Beta cut-off
        return column, value


# ============================================================
#  Choice and Implementation of the Static Evaluation Function
# ============================================================

def score_position(board, piece):
    """Evaluate the board state to assign a score"""
    score = 0

    # Score center column higher (central control is advantageous)
    center_array = [int(i) for i in list(board[:, COLUMN_COUNT // 2])]
    center_count = center_array.count(piece)
    score += center_count * 3

    # Score Horizontal
    for r in range(ROW_COUNT):
        row_array = [int(i) for i in list(board[r,:])]
        for c in range(COLUMN_COUNT - 3):
            window = row_array[c:c + WINDOW_LENGTH]
            score += evaluate_window(window, piece)

    # Score Vertical
    for c in range(COLUMN_COUNT):
        col_array = [int(i) for i in list(board[:, c])]
        for r in range(ROW_COUNT - 3):
            window = col_array[r:r + WINDOW_LENGTH]
            score += evaluate_window(window, piece)

    # Score positively sloped diagonals
    for r in range(ROW_COUNT - 3):
        for c in range(COLUMN_COUNT - 3):
            window = [board[r + i][c + i] for i in range(WINDOW_LENGTH)]
            score += evaluate_window(window, piece)

    # Score negatively sloped diagonals
    for r in range(ROW_COUNT - 3):
        for c in range(COLUMN_COUNT - 3):
            window = [board[r + 3 - i][c + i] for i in range(WINDOW_LENGTH)]
            score += evaluate_window(window, piece)

    return score

def evaluate_window(window, piece):
    """Evaluate a 4-item window for scoring"""
    score = 0
    opp_piece = PLAYER_PIECE if piece == AI_PIECE else AI_PIECE

    if window.count(piece) == 4:
        score += 100
    elif window.count(piece) == 3 and window.count(EMPTY) == 1:
        score += 5
    elif window.count(piece) == 2 and window.count(EMPTY) == 2:
        score += 2

    if window.count(opp_piece) == 3 and window.count(EMPTY) == 1:
        score -= 4

    return score


# ============================================================
#  Main game function to play the game
# ============================================================

def play_game():
    """Main function to start and control the game flow"""
    board = create_board()
    print_board(board)
    game_over = False
    turn = 0  # 0 for Player, 1 for AI

    while not game_over:
        # Player 1 Input
        if turn == 0:
            col = int(input("Player 1, make your selection (0-6): "))
            if is_valid_location(board, col):
                row = get_next_open_row(board, col)
                drop_piece(board, row, col, PLAYER_PIECE)

                if winning_move(board, PLAYER_PIECE):
                    print("Player 1 Wins!")
                    game_over = True

        # AI Input
        else:
            col, minimax_score = minimax_alpha_beta(board, 5, -np.inf, np.inf, True)
            if is_valid_location(board, col):
                row = get_next_open_row(board, col)
                drop_piece(board, row, col, AI_PIECE)

                if winning_move(board, AI_PIECE):
                    print("AI Wins!")
                    game_over = True

        print_board(board)
        turn += 1
        turn = turn % 2  # Alternate turns


# ============================================================
#  Run the game
# ============================================================
play_game()


[[0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]]


Player 1, make your selection (0-6):  1


[[0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]]
[[0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 2. 0. 0. 0.]]


Player 1, make your selection (0-6):  1


[[0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 2. 0. 0. 0.]]
[[0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 2. 0. 0. 0.]
 [0. 1. 0. 2. 0. 0. 0.]]


...

[[0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 2. 0. 0. 0.]
 [0. 2. 0. 2. 0. 0. 0.]
 [0. 1. 0. 1. 0. 0. 0.]
 [0. 1. 0. 2. 0. 0. 0.]
 [0. 1. 1. 2. 0. 0. 1.]]
[[0. 0. 0. 2. 0. 0. 0.]
 [0. 0. 0. 2. 0. 0. 0.]
 [0. 2. 0. 2. 0. 0. 0.]
 [0. 1. 0. 1. 0. 0. 0.]
 [0. 1. 0. 2. 0. 0. 0.]
 [0. 1. 1. 2. 0. 0. 1.]]


Player 1, make your selection (0-6):  2


[[0. 0. 0. 2. 0. 0. 0.]
 [0. 0. 0. 2. 0. 0. 0.]
 [0. 2. 0. 2. 0. 0. 0.]
 [0. 1. 0. 1. 0. 0. 0.]
 [0. 1. 1. 2. 0. 0. 0.]
 [0. 1. 1. 2. 0. 0. 1.]]
[[0. 0. 0. 2. 0. 0. 0.]
 [0. 0. 0. 2. 0. 0. 0.]
 [0. 2. 0. 2. 0. 0. 0.]
 [0. 1. 2. 1. 0. 0. 0.]
 [0. 1. 1. 2. 0. 0. 0.]
 [0. 1. 1. 2. 0. 0. 1.]]


Player 1, make your selection (0-6):  1


[[0. 0. 0. 2. 0. 0. 0.]
 [0. 1. 0. 2. 0. 0. 0.]
 [0. 2. 0. 2. 0. 0. 0.]
 [0. 1. 2. 1. 0. 0. 0.]
 [0. 1. 1. 2. 0. 0. 0.]
 [0. 1. 1. 2. 0. 0. 1.]]
AI Wins!
[[0. 0. 0. 2. 0. 0. 0.]
 [0. 1. 0. 2. 0. 0. 0.]
 [0. 2. 0. 2. 0. 0. 0.]
 [0. 1. 2. 1. 0. 0. 0.]
 [0. 1. 1. 2. 0. 0. 0.]
 [0. 1. 1. 2. 2. 0. 1.]]


In [ ]:
21
